In [ ]:
#### Use to merge different tiles into one .tif file
import glob
import rasterio
from rasterio.merge import merge
import os

from configs.config_object import ConfigObject

In [ ]:
config = ConfigObject('../config.json')
raw_bathy_path = config.paths.raw_bathy_path #input

In [ ]:
output_tif = raw_bathy_path
# !!! the merged files will be saved in raw_bathy_path. Be careful to not unintentionally overwrite a file.

In [ ]:

#### Find all .asc files
asc_files = glob.glob(r"C:\Users\leroquan\Documents\Data\mitgcm_grids\zug\swissbathy3d_zugersee_2056_5728.esriasciigrid\*.asc")

#### Open all the .asc files with rasterio
src_files_to_mosaic = []
for fp in asc_files:
    src = rasterio.open(fp)
    src_files_to_mosaic.append(src)

#### Merge them into a single mosaic
mosaic, out_trans = merge(src_files_to_mosaic)

#### Copy metadata from one of the source files and update
out_meta = src_files_to_mosaic[0].meta.copy()
out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_trans,
    "dtype": mosaic.dtype
})

#### Write the mosaic to disk
with rasterio.open(output_tif, "w", **out_meta) as dest:
    dest.write(mosaic)

print(f"Merged {len(asc_files)} .asc files into {output_tif}")